# データを学習・開発・評価に正しく分割する

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/01_data_io/data_split.ipynb)

NLP クックブック 01-07 ／ データの収集と入出力

データをランダムに分けるだけでは、クラスの偏りや、同じ文書・同じ書き手由来のデータが両側に入るリークが起きる。適切な分割をしたい。

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit, StratifiedGroupKFold, train_test_split

rng = np.random.default_rng(0)
n = 1000
df = pd.DataFrame({
    "text": [f"文 {i}" for i in range(n)],
    "label": rng.choice(["pos", "neg", "neu"], n, p=[0.6, 0.3, 0.1]),
    "author": rng.integers(0, 50, n),          # 同じ書き手の文が複数ある (学習者コーパス、レビュー、対話など)
})

# 1. 層化分割: クラスの比率を各分割で保つ (8:1:1)
train, rest = train_test_split(df, test_size=0.2, stratify=df["label"], random_state=0)
dev, test = train_test_split(rest, test_size=0.5, stratify=rest["label"], random_state=0)
print(pd.DataFrame({name: d["label"].value_counts(normalize=True).round(3) for name, d in [("train", train), ("dev", dev), ("test", test)]}))
print("書き手の重なり (train と test):", len(set(train["author"]) & set(test["author"])), "人")

       train   dev  test
label                   
pos    0.581  0.58  0.58
neg    0.322  0.32  0.33
neu    0.096  0.10  0.09
書き手の重なり (train と test): 44 人


In [2]:
# 2. グループ分割: 同じ書き手のデータが学習と評価にまたがらないようにする
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=0)
train_idx, test_idx = next(gss.split(df, groups=df["author"]))
print("書き手の重なり:", len(set(df["author"].iloc[train_idx]) & set(df["author"].iloc[test_idx])), "人")

# 3. 層化 + グループの交差検証
cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=0)
for fold, (tr, te) in enumerate(cv.split(df, df["label"], groups=df["author"])):
    print(fold, len(tr), len(te), df["label"].iloc[te].value_counts(normalize=True).round(2).to_dict())

書き手の重なり: 0 人
0 786 214 {'pos': 0.57, 'neg': 0.32, 'neu': 0.11}
1 808 192 {'pos': 0.58, 'neg': 0.33, 'neu': 0.09}
2 800 200 {'pos': 0.57, 'neg': 0.32, 'neu': 0.1}
3 799 201 {'pos': 0.6, 'neg': 0.32, 'neu': 0.08}
4 807 193 {'pos': 0.59, 'neg': 0.33, 'neu': 0.09}


In [3]:
# 分割結果は ID で保存して固定する (毎回ランダムに分け直さない)
df["split"] = "train"
df.loc[df.index[test_idx], "split"] = "test"
df[["text", "split"]].head(3)

,text,split
0,文 0,train
1,文 1,train
2,文 2,train


- 同じ文書から切り出した文、同じ人が書いた文、同じ原文の翻訳や言い換えは、必ず同じ側に入れる。両側にまたがると評価が不当に高くなる。
- 時系列のあるデータ（ニュース、SNS）は、ランダムではなく時間で分割する（過去で学習、未来で評価）。
- test は最後に 1 回だけ使う。ハイパーパラメータの調整やモデル選択は dev で行う。